# E07 · Homographies, stereo depth and convolution

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/06_estimation_perception/E07_homographies_stereo_depth_and_convolution.ipynb)

**Track:** Estimation and perception · **Skill axis (sheet):** Estimation and perception · **Time:** about 60 min

**Skill Hub concepts:** Homography; Stereo disparity; Convolution

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("E07")

## Map: three workhorses of robot vision

**Homography:** points on a plane (a chessboard, the floor) map between two images by a $3\times3$ matrix with 8 degrees
of freedom: $s\,[u', v', 1]^\top = H\,[u, v, 1]^\top$. Direct linear transform (DLT): each correspondence gives two rows
$$\begin{bmatrix} -u & -v & -1 & 0 & 0 & 0 & u u' & v u' & u' \\ 0 & 0 & 0 & -u & -v & -1 & u v' & v v' & v' \end{bmatrix} h = 0$$
Stack $\ge 4$ correspondences, take the right singular vector of the smallest singular value, reshape to $3\times3$,
divide by $h_{33}$. (Normalizing points first improves conditioning; RANSAC rejects bad matches.)

**Stereo:** two rectified cameras a baseline $B$ apart; a point shifts by disparity $d$ pixels:
$$Z = \frac{f\,B}{d}, \qquad \Delta Z \approx \frac{Z^2}{f B}\,\Delta d \quad \text{(error grows with the square of the distance)}$$

**Convolution:** $(I * k)(x, y) = \sum_{i,j} I(x - i, y - j)\,k(i, j)$: the kernel is **flipped**; without the flip it is
correlation (what most deep learning libraries call "convolution"). "Valid" output size: $(H - k_h + 1) \times (W - k_w + 1)$.

In [ ]:
import numpy as np
import cv2

## Task 1: estimate a homography with the DLT

In [ ]:
def dlt_homography(src, dst):
    """src, dst: (N, 2) pixel arrays, N >= 4. Return H (3x3) with H[2, 2] = 1."""
    raise NotImplementedError  # TODO

lab.check("homography", dlt_homography)

Use it like Gambit would: rectify a tilted view of a chessboard to a top-down 400x400 image.

In [ ]:
try:
    board = np.zeros((400, 400), np.uint8)
    for i in range(8):
        for j in range(8):
            if (i + j) % 2 == 0:
                board[i * 50:(i + 1) * 50, j * 50:(j + 1) * 50] = 255
    corners_top = np.array([[0, 0], [399, 0], [399, 399], [0, 399]], float)
    corners_cam = np.array([[120, 80], [500, 110], [560, 420], [60, 380]], float)
    camera_view = cv2.warpPerspective(board, dlt_homography(corners_top, corners_cam), (640, 480))
    rectified = cv2.warpPerspective(camera_view, dlt_homography(corners_cam, corners_top), (400, 400))
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(9, 4)); ax[0].imshow(camera_view, cmap="gray"); ax[0].set_title("camera")
    ax[1].imshow(rectified, cmap="gray"); ax[1].set_title("rectified top-down"); plt.show()
except NotImplementedError:
    print("finish Task 1 first")

## Task 2: depth from disparity

In [ ]:
def depth_from_disparity(d_px, f_px, baseline_m):
    """Return depth in meters; np.inf where disparity <= 0."""
    raise NotImplementedError  # TODO
    

lab.check("stereo", depth_from_disparity)

## Task 3: true 2D convolution, valid mode, with loops or slicing (no library calls)

In [ ]:
def conv2d_valid(img, k):
    raise NotImplementedError  # TODO

lab.check("convolution", conv2d_valid)

## Checkpoint: Math
**M1 (Apply).** $f = 700$ px, baseline 0.12 m, disparity 35 px. Depth in meters?

**M2 (Remember).** Degrees of freedom of a homography?

**M3 (Apply).** Valid convolution of a 64 x 64 image with a 5 x 5 kernel. Output width?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** Your stereo camera gives depth accurate to 1 cm at 0.5 m. Roughly how accurate at 2 m, all else equal?
- a) 4 cm
- b) 16 cm (error grows with distance squared)
- c) 1 cm

**K2 (Understand).** When is a homography the right model between two images and when is it wrong? Use the chessboard
and a cluttered table as examples.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `e07.h`).
3. In `roboprep/checks/e07.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()